# Agent Types

NAT provides several agent types optimized for different use cases. This tutorial covers the three main agent types and when to use each.

## What You'll Learn

1. ReAct Agent - Reasoning and acting with explicit thought process
2. Tool Calling Agent - Direct function calling without explicit reasoning
3. ReWOO Agent - Planning-first approach for complex tasks
4. Choosing the right agent for your use case

## Agent Comparison

| Agent | Best For | Pros | Cons |
|-------|----------|------|------|
| **ReAct** | General tasks, debugging | Transparent reasoning, flexible | More tokens used |
| **Tool Calling** | Simple tasks, low latency | Fast, efficient | Less explainable |
| **ReWOO** | Complex multi-step tasks | Better planning | Higher initial latency |


In [ ]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## Setup: Common Components

First, let's create the LLM and tools we'll use for all agent examples:


In [ ]:
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
    print("✅ LLM and tools created (with calculator)")
except ImportError:
    tools = [time_tool]
    print("✅ LLM and tools created (time only)")


## 1. ReAct Agent

The ReAct (Reasoning + Acting) agent explicitly shows its thought process before taking actions. This makes it excellent for debugging and understanding agent behavior.

### Key Features
- Explicit "Thought" steps before actions
- Clear reasoning chain
- Good for complex reasoning tasks
- More verbose output


In [ ]:
from nat.agent.react_agent.register import NatReActAgent

# Create ReAct Agent
react_agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,                           # Show reasoning
    parse_agent_response_max_retries=3,     # Retry on parse errors
    additional_instructions="Think step by step before answering.",
)

react_workflow = NatWorkflow(entrypoint=react_agent)
print("✅ ReAct Agent created")


In [ ]:
# Save ReAct config
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

react_config = config_dir / "react_agent.yaml"
react_workflow.save_to_config_file(react_config)
print(f"📄 Saved to: {react_config}")


## 2. Tool Calling Agent

The Tool Calling agent uses the LLM's native function calling capabilities. It's faster and more efficient but provides less insight into the reasoning process.

### Key Features
- Native LLM function calling
- Lower latency
- Works well with models trained for tool use
- Less verbose output


In [ ]:
from nat.agent.tool_calling_agent.register import ToolCallingAgent

# Create Tool Calling Agent
tool_calling_agent = ToolCallingAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=10,  # Maximum tool calls before stopping
)

tool_calling_workflow = NatWorkflow(entrypoint=tool_calling_agent)
print("✅ Tool Calling Agent created")


In [ ]:
# Save Tool Calling config
tool_calling_config = config_dir / "tool_calling_agent.yaml"
tool_calling_workflow.save_to_config_file(tool_calling_config)
print(f"📄 Saved to: {tool_calling_config}")


## 3. ReWOO Agent

The ReWOO (Reasoning WithOut Observation) agent creates a plan upfront before executing any tools. This is useful for complex multi-step tasks where planning ahead is beneficial.

### Key Features
- Plans all steps before execution
- Better for complex multi-step tasks
- Reduces back-and-forth with LLM
- Higher initial latency, potentially faster overall


In [ ]:
from nat.agent.rewoo_agent.register import ReWOOAgentWorkflow

# Create ReWOO Agent
rewoo_agent = ReWOOAgentWorkflow(
    tools=tools,
    llm=llm,
    verbose=True,
)

rewoo_workflow = NatWorkflow(entrypoint=rewoo_agent)
print("✅ ReWOO Agent created")


In [ ]:
# Save ReWOO config
rewoo_config = config_dir / "rewoo_agent.yaml"
rewoo_workflow.save_to_config_file(rewoo_config)
print(f"📄 Saved to: {rewoo_config}")


## Running the Agents

### Via CLI

```bash
# ReAct Agent
nat run --config_file configs/react_agent.yaml --input "What is 25 * 4?"

# Tool Calling Agent  
nat run --config_file configs/tool_calling_agent.yaml --input "What is 25 * 4?"

# ReWOO Agent
nat run --config_file configs/rewoo_agent.yaml --input "What is 25 * 4?"
```

### Via Python


In [ ]:
# Test each agent (uncomment to run)
# result = await react_workflow.prompt("What is 25 * 4?")
# result = await tool_calling_workflow.prompt("What is 25 * 4?")
# result = await rewoo_workflow.prompt("What is 25 * 4?")


## Choosing the Right Agent

### Use ReAct When:
- You need to debug agent behavior
- Tasks require complex reasoning
- Transparency is important
- You want to see the agent's thought process

### Use Tool Calling When:
- Speed is critical
- Tasks are straightforward
- Using models optimized for function calling
- You don't need detailed reasoning logs

### Use ReWOO When:
- Tasks have multiple dependent steps
- Planning ahead would reduce errors
- You want to minimize LLM round-trips
- Complex workflows with clear sub-tasks

## Summary

✅ **ReAct** - Explicit reasoning, great for debugging  
✅ **Tool Calling** - Fast and efficient for simple tasks  
✅ **ReWOO** - Planning-first for complex multi-step tasks  

## Next Steps

- **[04_functions_and_tools.ipynb](./04_functions_and_tools.ipynb)** - Create custom functions
- **[05_llms.ipynb](./05_llms.ipynb)** - Configure different LLM providers
